# Teste RAG simples para a coleção `3gpp_rel18_all`

Notebook mínimo para conectar ao Qdrant, carregar embeddings, executar uma busca k-NN e gerar uma resposta com um placeholder de LLM.

## 1. Instalar dependências

Execute este comando no ambiente se ainda não tiver instalado as dependências necessárias.

In [ ]:
!pip install qdrant-client sentence-transformers torch transformers openai

## 2. Importar bibliotecas e configurar variáveis

In [ ]:
import os
import numpy as np
from typing import List, Dict, Any

from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

# Ajuste conforme seu ambiente de execução
QDRANT_HOST = os.getenv('QDRANT_HOST', 'localhost')
QDRANT_PORT = int(os.getenv('QDRANT_PORT', '6333'))
COLLECTION_NAME = os.getenv('COLLECTION_NAME', '3gpp_rel18_all')
DEVICE = os.getenv('EMB_DEVICE', 'cpu')  # 'cpu' ou 'cuda'

## 3. Conectar ao Qdrant e checar coleção

In [ ]:
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

try:
    collection_info = client.get_collection(COLLECTION_NAME)
    print(f"Coleção '{COLLECTION_NAME}' existe.")
    count = client.count(COLLECTION_NAME).count
    print(f"Pontos na coleção: {count}")
except Exception as e:
    print(f"Erro ao acessar coleção '{COLLECTION_NAME}': {e}")

## 4. Carregar modelo de embeddings

In [ ]:
print('Carregando modelo de embeddings...')
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEVICE)
print('Modelo carregado.')

## 5. Função auxiliar: calcular embeddings

In [ ]:
def embed_texts(texts: List[str], batch_size: int = 32):
    if len(texts) == 0:
        return np.empty((0, embed_model.get_sentence_embedding_dimension()))
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        emb = embed_model.encode(batch, convert_to_numpy=True, show_progress_bar=False, normalize_embeddings=True)
        embeddings.append(emb)
    return np.vstack(embeddings)

## 6. Recuperação (k-NN) e montagem de contexto

In [ ]:
def search_qdrant(query: str, k: int = 3):
    query_embedding = embed_texts([query])[0]
    results = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_embedding.tolist(),
        limit=k,
        with_payload=True,
        with_vector=True,
    )
    return results

def build_context(results):
    lines = []
    for hit in results:
        payload = hit.payload or {}
        text = payload.get('text', payload.get('page_content', '<sem texto>'))
        meta = {k: v for k, v in payload.items() if k != 'text'}
        lines.append(f"----
Metadata: {meta}
{text}
,
]},{
:
,
:{},
:[
7
,
,
]},{
:
,
:{},
:[
,
,
,
    if OPENAI_API_KEY:
        response = openai.Completion.create(
            model='text-davinci-003',
            prompt=prompt,
            max_tokens=200,
            temperature=0.3,
        )
        return response.choices[0].text.strip()
    return f"[LLM placeholder] Pergunta: {query}
Contexto extraído: {context[:500]}..."

## 8. Exemplo interativo: consulta e saída

In [ ]:
user_query = input('Pergunta para a coleção 3gpp_rel18_all: ')
results = search_qdrant(user_query, k=3)
context = build_context(results)
answer = generate_answer(user_query, context)
print('
=== Resposta ===')
print(answer)
print('
=== Trechos recuperados ===')
for idx, hit in enumerate(results, 1):
    print(f'Hit {idx}: score={hit.score}')",
,
]},{
:
,
:{},
:[
9